# LoRA GPT-2 Medium E2E Training on Google Colab

Use this notebook to run the GPT-2 Medium + LoRA + E2E NLG replication on a Colab GPU.

Recommended runtime:

- Runtime > Change runtime type > Hardware accelerator: GPU
- A T4 GPU should be enough for the baseline, though full training can still take several hours.

The notebook clones the project from GitHub, downloads the E2E files, preprocesses them, runs a dry run, then starts training.

## 1. Check GPU

Run this first to confirm Colab assigned a CUDA GPU.

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone The Repo

If you rerun the notebook in the same runtime, this cell updates the existing clone instead of failing.

In [ ]:
from getpass import getpass

token = getpass("GitHub token: ")

In [ ]:
%cd /content
!git clone https://$token@github.com/justinlxiang/CS4782-final-project.git
%cd /content/CS4782-final-project/lora-gpt2-medium-e2e
!pwd

## 3. Install Dependencies

Colab already has PyTorch installed. Installing the requirements should add the remaining packages.

In [ ]:
!pip install -q -r requirements.txt

## 4. Optional: Mount Google Drive

Recommended for long runs. Colab runtimes can disconnect, so copying checkpoints to Drive protects your outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/e2e_lora_r4_alpha32"

## 5. Download E2E Dataset

These files come from the official Microsoft LoRA repo and are already formatted as `context||completion`.

In [ ]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt

!wc -l data/raw/e2e/*.txt

## 6. Preprocess E2E

This creates tokenized JSONL files under `data/processed/e2e_gpt2/`.

The current preprocessing uses the official-style sequence:

`raw_context + 50256 + leading_space_completion + 50256`

In [ ]:
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml

import json
from pathlib import Path

path = Path("data/processed/e2e_gpt2/train.jsonl")
example = json.loads(path.read_text().splitlines()[0])

print("prompt:", example["prompt"])
print("first input ids:", example["input_ids"][:20])
print("first labels:", example["labels"][:20])
print("prompt length:", example["prompt_length"])

## 7. Run Tests And Dry Run

This confirms the code works, LoRA injects into all 24 GPT-2 Medium layers, CUDA works, and a tiny forward pass succeeds.

In [ ]:
!python -m pytest
!python scripts/count_params.py --config configs/e2e_gpt2_medium_lora.yaml
!python scripts/train.py --config configs/e2e_gpt2_medium_lora.yaml --dry-run --device cuda --dry-run-forward-pass

## 8. Optional: Short Smoke Training

This runs a few real optimizer steps to confirm backward pass and optimizer updates work on CUDA. It is not the full experiment.

In [ ]:
!python scripts/train.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --smoke-train \
  --device cuda \
  --dry-run-max-examples 80 \
  --dry-run-batch-size 8 \
  --smoke-max-steps 10

## 9. Full Training

This is the paper-style baseline run: GPT-2 Medium, E2E, LoRA rank 4, alpha 32, dropout 0.1, batch size 8, 5 epochs.

Outputs go to:

`outputs/runs/e2e_lora_r4_alpha32/`

Checkpoints are saved every 1000 steps and at the end.

In [ ]:
!python scripts/train.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --train \
  --device cuda

## 10. Back Up Outputs To Drive

Run this after training, or periodically if you are worried about Colab disconnects.

In [ ]:
!mkdir -p /content/drive/MyDrive
!cp -r outputs/runs/e2e_lora_r4_alpha32 "$DRIVE_OUTPUT_DIR"
!du -sh "$DRIVE_OUTPUT_DIR"

## 11. Resume From A Checkpoint

If Colab disconnects, point to the latest checkpoint and resume. A resumed run appends to the same `metrics.jsonl` and skips already-completed batches from the checkpointed epoch.

In [ ]:
# Example: change this to the latest checkpoint you have.
RESUME_CHECKPOINT = "outputs/runs/e2e_lora_r4_alpha32/checkpoints/adapter_step_1000.pt"

!python scripts/train.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --train \
  --device cuda \
  --resume-checkpoint "$RESUME_CHECKPOINT"

## 12. Inspect Training Logs

Use this to check the final few loss records and available checkpoints.

In [ ]:
!tail -n 10 outputs/runs/e2e_lora_r4_alpha32/metrics.jsonl
!ls -lh outputs/runs/e2e_lora_r4_alpha32/checkpoints | tail